In [88]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"


#IMPORTS

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer

from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from imblearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE


In [89]:
#charger data (sans et avec clusters)
path_raw = "../data/raw/rawFile.csv"

path_clusters = "../data/processed/clusters_data.csv"

df = pd.read_csv(path_raw)

df_clusters = pd.read_csv(
    path_clusters,
    usecols=["customerID", "Cluster"]
)

print("****** Shape des donnees originales ******")
print(df.shape)


****** Shape des donnees originales ******
(7043, 21)


In [90]:
#AJOUT DU CLUSTER

df_clusters = df_clusters.drop_duplicates(
    subset=["customerID"]
)

df = df.merge(
    df_clusters,
    on="customerID",
    how="left",
    validate="one_to_one"
)

if df["Cluster"].isna().any():
    raise ValueError(
        "Certains clients n'ont pas de cluster correspondant."
    )


# cluster as categoriell non numeric
df["Cluster"] = df["Cluster"].astype(str)



In [91]:
#preparation des donnees

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)


# Feature Engineering :
# calcul du cout moyen mensuel approximatif.

df["AvgMonthlySpend"] = (
    df["TotalCharges"]
    /
    df["tenure"].replace(0, 1)
)


# Variable cible
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})


# Variables initiales :
# sans Churn, customerID, AvgMonthlySpend et Cluster.

X_base = df.drop(
    columns=[
        "Churn",
        "customerID",
        "AvgMonthlySpend",
        "Cluster"
    ],
    errors="ignore"
)


# Variables initiales + Feature Engineering

X_fe = X_base.copy()

X_fe["AvgMonthlySpend"] = df["AvgMonthlySpend"]


# Variables initiales + Feature Engineering + Cluster

X_fe_cluster = X_fe.copy()

X_fe_cluster["Cluster"] = df["Cluster"]


print("****** Comparaison des features ******")

print("Features initiales :", X_base.shape[1])

print("Features avec Feature Engineering :", X_fe.shape[1])

print("Features avec Feature Engineering + Cluster :",
      X_fe_cluster.shape[1])



****** Comparaison des features ******
Features initiales : 19
Features avec Feature Engineering : 20
Features avec Feature Engineering + Cluster : 21


In [92]:

#TRAIN / TEST SPLIT

X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


train_idx = X_train_base.index
test_idx = X_test_base.index


X_train_fe = X_fe.loc[train_idx]
X_test_fe = X_fe.loc[test_idx]


X_train_fe_cluster = X_fe_cluster.loc[train_idx]
X_test_fe_cluster = X_fe_cluster.loc[test_idx]


print("****** TRAIN / TEST ******")

print("X_train :", X_train_base.shape)
print("X_test :", X_test_base.shape)

print("y_train :", y_train.shape)
print("y_test :", y_test.shape)



****** TRAIN / TEST ******
X_train : (5634, 19)
X_test : (1409, 19)
y_train : (5634,)
y_test : (1409,)


In [93]:

#FONCTION DE PREPROCESSING

def create_preprocessor(X_train_data):

    numeric_features = X_train_data.select_dtypes(
        include=["int64", "float64"]
    ).columns

    categorical_features = X_train_data.select_dtypes(
        include=["object", "category"]
    ).columns


    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )


    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_features
            ),
            (
                "cat",
                categorical_pipeline,
                categorical_features
            )
        ]
    )

    return preprocessor



In [ ]:

#CREATION DU PIPELINE

def create_model_pipeline(X_train_data, model):

    preprocessor = create_preprocessor(
        X_train_data
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "smote",
                SMOTE(
                    random_state=42
                )
            ),
            (
                "model",
                model
            )
        ]
    )

    return pipeline

In [95]:
#FONCTION D'EVALUATION
def evaluate_model(
    pipeline,
    X_test,
    y_test,
    scenario
):
    #prediction de la classe
    y_pred= pipeline.predict(
        X_test
    )
    y_proba= pipeline.predict_proba(
        X_test
    )[:,1]

   

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_proba
    )


    print("******", scenario, "******")

    print("Precision :", round(precision, 4))
    print("Recall    :", round(recall, 4))
    print("F1-score  :", round(f1, 4))
    print("ROC-AUC   :", round(roc_auc, 4))


    return {
        "Scenario": scenario,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    }



In [96]:

#ETAPE 1 : MODELE DE BASE AVANT OPTIMISATION

baseline_model = GradientBoostingClassifier(
    random_state=42
)


baseline_pipeline = create_model_pipeline(
    X_train_base,
    baseline_model
)


baseline_pipeline.fit(
    X_train_base,
    y_train
)


baseline_result = evaluate_model(
    baseline_pipeline,
    X_test_base,
    y_test,
    "Avant optimisation"
)



****** Avant optimisation ******
Precision : 0.5575
Recall    : 0.6738
F1-score  : 0.6102
ROC-AUC   : 0.8395


In [97]:

#ETAPE 2 : VALIDATION CROISEE

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


#ETAPE 3 : HYPERPARAMETER TUNING

param_distributions = {

    "model__n_estimators": [
        100, 150, 200, 250
    ],

    "model__learning_rate": [
        0.03, 0.05, 0.1
    ],

    "model__max_depth": [
        1, 2, 3
    ],

    "model__min_samples_split": [
        2, 5, 10
    ],

    "model__min_samples_leaf": [
        1, 2, 4
    ],

    "model__subsample": [
        0.8, 1.0
    ]
}


search = RandomizedSearchCV(
    estimator=create_model_pipeline(
        X_train_base,
        GradientBoostingClassifier(
            random_state=42
        )
    ),
    param_distributions=param_distributions,
    n_iter=12,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    refit=True,
    verbose=1
)


search.fit(
    X_train_base,
    y_train
)


print("****** Meilleurs hyperparamètres ******")
print(search.best_params_)


print("****** Meilleur F1 moyen en validation croisée ******")
print(round(search.best_score_, 4))


# Meilleur pipeline entraîné sur tout le train
# avec les hyperparamètres sélectionnés.

best_pipeline = search.best_estimator_


Fitting 5 folds for each of 12 candidates, totalling 60 fits
****** Meilleurs hyperparamètres ******
{'model__subsample': 1.0, 'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_depth': 3, 'model__learning_rate': 0.03}
****** Meilleur F1 moyen en validation croisée ******
0.637


In [98]:


#ETAPE 4 : MODELE APRES OPTIMISATION

optimized_result = evaluate_model(
    best_pipeline,
    X_test_base,
    y_test,
    "Après optimisation"
)


# Récupérer les parametres selectionnes.

best_model_params = {
    key.replace("model__", ""): value
    for key, value in search.best_params_.items()
}


#ETAPE 5 : APPORT DU FEATURE ENGINEERING

model_fe = GradientBoostingClassifier(
    random_state=42,
    **best_model_params
)


pipeline_fe = create_model_pipeline(
    X_train_fe,
    model_fe
)


pipeline_fe.fit(
    X_train_fe,
    y_train
)


fe_result = evaluate_model(
    pipeline_fe,
    X_test_fe,
    y_test,
    "Après optimisation + Feature Engineering"
)


****** Après optimisation ******
Precision : 0.5565
Recall    : 0.7112
F1-score  : 0.6244
ROC-AUC   : 0.8422
****** Après optimisation + Feature Engineering ******
Precision : 0.5537
Recall    : 0.7032
F1-score  : 0.6196
ROC-AUC   : 0.8416


In [99]:


#ETAPE 6 : APPORT DU CLUSTER

model_fe_cluster = GradientBoostingClassifier(
    random_state=42,
    **best_model_params
)


pipeline_fe_cluster = create_model_pipeline(
    X_train_fe_cluster,
    model_fe_cluster
)


pipeline_fe_cluster.fit(
    X_train_fe_cluster,
    y_train
)


fe_cluster_result = evaluate_model(
    pipeline_fe_cluster,
    X_test_fe_cluster,
    y_test,
    "Après optimisation + Feature Engineering + Cluster"
)



****** Après optimisation + Feature Engineering + Cluster ******
Precision : 0.5549
Recall    : 0.7032
F1-score  : 0.6203
ROC-AUC   : 0.8414


In [100]:

#COMPARAISON FINALE

results = pd.DataFrame([
    baseline_result,
    optimized_result,
    fe_result,
    fe_cluster_result
])



print(" ***************COMPARAISON DES EXPERIENCES**********")

print(
    results.round(4).to_string(
        index=False
    )
)



 ***************COMPARAISON DES EXPERIENCES**********
                                          Scenario  Precision  Recall  F1-score  ROC-AUC
                                Avant optimisation     0.5575  0.6738    0.6102   0.8395
                                Après optimisation     0.5565  0.7112    0.6244   0.8422
          Après optimisation + Feature Engineering     0.5537  0.7032    0.6196   0.8416
Après optimisation + Feature Engineering + Cluster     0.5549  0.7032    0.6203   0.8414


In [101]:

#IMPACT DE CHAQUE AMELIORATION

baseline = results.iloc[0]
optimized = results.iloc[1]
with_fe = results.iloc[2]
with_cluster = results.iloc[3]


metrics = [
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
]


delta_results = []


for metric in metrics:

    delta_results.append({

        "Metric": metric,

        "Gain_Optimisation": (
            optimized[metric]
            -
            baseline[metric]
        ),

        "Gain_Feature_Engineering": (
            with_fe[metric]
            -
            optimized[metric]
        ),

        "Gain_Cluster": (
            with_cluster[metric]
            -
            with_fe[metric]
        )

    })


delta_df = pd.DataFrame(
    delta_results
)


print("*******IMPACT DES AMELIORATIONS**********")
print(
    delta_df.round(4).to_string(
        index=False
    )
)


*******IMPACT DES AMELIORATIONS**********
   Metric  Gain_Optimisation  Gain_Feature_Engineering  Gain_Cluster
Precision            -0.0010                   -0.0028        0.0012
   Recall             0.0374                   -0.0080        0.0000
 F1-score             0.0142                   -0.0049        0.0007
  ROC-AUC             0.0027                   -0.0006       -0.0002


In [102]:


#SAUVEGARDE

results.to_csv(
    "../data/processed/optimization_results.csv",
    index=False
)


delta_df.to_csv(
    "../data/processed/optimization_comparison.csv",
    index=False
)


pd.DataFrame(
    search.cv_results_
).to_csv(
    "../data/processed/optimization_cv_results.csv",
    index=False
)


print("optimization_results.csv créé avec succès !")

print("optimization_comparison.csv créé avec succès !")

print("optimization_cv_results.csv créé avec succès !")

optimization_results.csv créé avec succès !
optimization_comparison.csv créé avec succès !
optimization_cv_results.csv créé avec succès !
